# 04 — Build tracked pose sequences using the frozen group split

This reads only `Notebook_2_Improved_v2/data/tracked_keypoints_v2/*.npz` and reuses Notebook 1's `video_split_manifest.csv` and `label_map.json`. It never reassigns videos or recording groups.

A new timestamped output folder is created on every run. Missing tracked videos are reported; pilot datasets are explicitly marked partial.

In [ ]:
from pathlib import Path
from datetime import datetime
import json, gc, hashlib
import numpy as np
import pandas as pd

PROJECT_DIR=Path(r"C:\Users\admin\ViTPose_Action_Recognition")
NOTEBOOK2_DIR=PROJECT_DIR/"Notebook_2_Improved_v2"
SOURCE_SEQ=PROJECT_DIR/"data"/"processed"/"sequences_group_split_v1"
TRACKED_DIR=NOTEBOOK2_DIR/"data"/"tracked_keypoints_v2"
OUT_ROOT=NOTEBOOK2_DIR/"data"/"sequences_tracked_group_split_v2"
OUT_ROOT.mkdir(parents=True,exist_ok=True)
RUN_ID=datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR=OUT_ROOT/RUN_ID
OUT_DIR.mkdir(parents=True,exist_ok=False)

SEQUENCE_LENGTH=16
SEQUENCE_STRIDE=4
MIN_VALID_FRAME_RATIO=0.60
MIN_VALID_JOINTS=8

SPLIT_PATH=SOURCE_SEQ/"video_split_manifest.csv"
LABEL_MAP_PATH=SOURCE_SEQ/"label_map.json"
assert SPLIT_PATH.is_file() and LABEL_MAP_PATH.is_file(),"Run Notebook 1 group-aware sequence builder first."
assert TRACKED_DIR.is_dir(),"Run Notebook 2 Notebook 03 first."
split_df=pd.read_csv(SPLIT_PATH)
label_map=json.loads(LABEL_MAP_PATH.read_text(encoding="utf-8"))
label_to_id={v:int(k) for k,v in label_map.items()}
required={"video_path","label","split","video_id","recording_group_key"}
missing=required-set(split_df.columns)
assert not missing,f"Split manifest missing: {sorted(missing)}"

# Hard assertion: do not permit any group reassignment or leakage.
group_sets={s:set(split_df.loc[split_df.split.eq(s),"recording_group_key"].astype(str)) for s in ["train","val","test"]}
for a,b in [("train","val"),("train","test"),("val","test")]:
    assert not group_sets[a]&group_sets[b],f"Recording-group overlap in source split: {a} / {b}"

def path_key(p):
    return str(Path(str(p)).resolve()).replace("/","\\").casefold()
lookup={path_key(r.video_path):r._asdict() for r in split_df.itertuples(index=False)}
tracked_files=sorted(p for p in TRACKED_DIR.glob("*.npz") if ".tmp." not in p.name and ".partial." not in p.name)
rows=[]; bad=[]
for p in tracked_files:
    try:
        with np.load(p,allow_pickle=False) as d:
            needed={"keypoints","boxes","frame_indices","has_person","track_ids","label","video_path","complete_video","decoded_frames","max_decoded_frames"}
            if not needed.issubset(d.files): raise ValueError(f"Missing keys: {sorted(needed-set(d.files))}")
            vp=str(d["video_path"].item()); label=str(d["label"].item())
            n=len(d["frame_indices"])
            complete_video=bool(d["complete_video"].item())
            decoded_frames=int(d["decoded_frames"]); max_decoded_frames=int(d["max_decoded_frames"])
            if d["keypoints"].shape!=(n,17,3) or d["boxes"].shape!=(n,4) or d["has_person"].shape!=(n,) or d["track_ids"].shape!=(n,):
                raise ValueError("Inconsistent tracked arrays")
        meta=lookup.get(path_key(vp))
        if meta is None: raise ValueError("Video path not found in frozen Notebook 1 group split manifest")
        if label!=str(meta["label"]): raise ValueError("Label mismatch with frozen split manifest")
        rows.append({"npz_path":str(p),"video_path":vp,"label":label,"split":str(meta["split"]),
                     "video_id":int(meta["video_id"]),"recording_group_key":str(meta["recording_group_key"]),"sampled_frames":n,
                     "complete_video":complete_video,"decoded_frames":decoded_frames,"max_decoded_frames":max_decoded_frames})
    except Exception as exc: bad.append({"npz_path":str(p),"error":repr(exc)})
tracked_df=pd.DataFrame(rows)
assert not tracked_df.empty,"No valid tracked NPZ files found. Run Notebook 03."
if bad: pd.DataFrame(bad).to_csv(OUT_DIR/"unreadable_tracked_npz.csv",index=False)
expected=set(split_df.video_path.map(path_key)); found=set(tracked_df.video_path.map(path_key))
missing_videos=sorted(expected-found)
incomplete_videos=tracked_df.loc[~tracked_df["complete_video"].astype(bool),"video_path"].tolist()
full_coverage=(len(missing_videos)==0 and len(incomplete_videos)==0)
pd.DataFrame({"video_path":missing_videos}).to_csv(OUT_DIR/"videos_missing_tracked_keypoints.csv",index=False)
print("Expected split videos:",len(expected),"| tracked videos:",len(found),"| coverage:",round(len(found)/max(1,len(expected)),4))
print("Full source-video coverage:",full_coverage,"| missing files:",len(missing_videos),"| partial clips:",len(incomplete_videos),"| unreadable NPZs:",len(bad))
if incomplete_videos:
    pd.DataFrame({"video_path":incomplete_videos}).to_csv(OUT_DIR/"videos_not_processed_to_eof.csv",index=False)

In [ ]:
def normalize_pose(keypoints,boxes,pose_ok):
    n=len(keypoints)
    out=np.zeros((n,17,3),dtype=np.float32)
    frame_valid=np.zeros(n,dtype=bool)
    for t in range(n):
        kp=np.asarray(keypoints[t],dtype=np.float32); box=np.asarray(boxes[t],dtype=np.float32)
        if not pose_ok[t] or not np.isfinite(box).all(): continue
        x1,y1,x2,y2=map(float,box); bw=x2-x1; bh=y2-y1
        if bw<2 or bh<2: continue
        finite_xy=np.isfinite(kp[:,:2]).all(axis=1)
        if int(finite_xy.sum())<MIN_VALID_JOINTS: continue
        # Normalize each joint relative to the selected person's bounding box.
        out[t,finite_xy,0]=(kp[finite_xy,0]-x1)/bw
        out[t,finite_xy,1]=(kp[finite_xy,1]-y1)/bh
        scores=np.nan_to_num(kp[:,2],nan=0.0,posinf=0.0,neginf=0.0)
        out[t,:,2]=np.clip(scores,0.0,1.0)
        # Missing joints are explicitly encoded as (0,0,confidence=0).
        out[t,~finite_xy,:2]=0.0
        out[t,~finite_xy,2]=0.0
        frame_valid[t]=True
    return out,frame_valid

def iter_video_windows(npz_path):
    with np.load(npz_path,allow_pickle=False) as d:
        kp=np.asarray(d["keypoints"],dtype=np.float32)
        bx=np.asarray(d["boxes"],dtype=np.float32)
        fi=np.asarray(d["frame_indices"],dtype=np.int32)
        hp=np.asarray(d["has_person"],dtype=bool)
    pose,valid=normalize_pose(kp,bx,hp)
    T=len(pose)
    if T<SEQUENCE_LENGTH: return
    starts=list(range(0,T-SEQUENCE_LENGTH+1,SEQUENCE_STRIDE))
    last=T-SEQUENCE_LENGTH
    if starts and starts[-1]!=last: starts.append(last)
    for s in starts:
        e=s+SEQUENCE_LENGTH
        ratio=float(valid[s:e].mean())
        if ratio<MIN_VALID_FRAME_RATIO: continue
        yield pose[s:e].reshape(SEQUENCE_LENGTH,51),int(fi[s]),int(fi[e-1]),ratio

counts={"train":0,"val":0,"test":0}
for i,row in enumerate(tracked_df.itertuples(index=False),1):
    counts[row.split]+=sum(1 for _ in iter_video_windows(row.npz_path))
    if i%500==0: print(f"Counted {i} / {len(tracked_df)} videos")
print("Windows by split:",counts)

In [ ]:
# Allocate disk-backed arrays; zero-window splits are saved as ordinary empty .npy arrays.
memmaps={}; offsets={s:0 for s in ["train","val","test"]}
for split in ["train","val","test"]:
    shape=(counts[split],SEQUENCE_LENGTH,51)
    if counts[split]>0:
        X=np.lib.format.open_memmap(OUT_DIR/f"X_{split}.npy",mode="w+",dtype=np.float32,shape=shape)
        y=np.lib.format.open_memmap(OUT_DIR/f"y_{split}.npy",mode="w+",dtype=np.int64,shape=(counts[split],))
        vids=np.lib.format.open_memmap(OUT_DIR/f"videos_{split}.npy",mode="w+",dtype=np.int32,shape=(counts[split],))
        memmaps[split]={"X":X,"y":y,"videos":vids}
    else:
        np.save(OUT_DIR/f"X_{split}.npy",np.empty(shape,dtype=np.float32))
        np.save(OUT_DIR/f"y_{split}.npy",np.empty((0,),dtype=np.int64))
        np.save(OUT_DIR/f"videos_{split}.npy",np.empty((0,),dtype=np.int32))

per_video=[]
for i,row in enumerate(tracked_df.itertuples(index=False),1):
    before=offsets[row.split]
    class_id=label_to_id[row.label]
    for window,start_frame,end_frame,ratio in iter_video_windows(row.npz_path):
        j=offsets[row.split]
        memmaps[row.split]["X"][j]=window
        memmaps[row.split]["y"][j]=class_id
        memmaps[row.split]["videos"][j]=int(row.video_id)
        offsets[row.split]+=1
    per_video.append({"video_id":int(row.video_id),"video_path":row.video_path,"label":row.label,
                      "split":row.split,"recording_group_key":row.recording_group_key,
                      "source_npz":row.npz_path,"sequence_windows":offsets[row.split]-before})
    if i%250==0: print(f"Wrote {i} / {len(tracked_df)} videos")
for split in memmaps:
    for arr in memmaps[split].values(): arr.flush()
del memmaps; gc.collect()

In [ ]:
summary=pd.DataFrame(per_video)
summary.to_csv(OUT_DIR/"video_sequence_summary.csv",index=False)
tracked_df.to_csv(OUT_DIR/"tracked_video_manifest.csv",index=False)
(OUT_DIR/"label_map.json").write_text(json.dumps(label_map,indent=2),encoding="utf-8")

# Save split manifests copied from Notebook 1; never regenerate or modify the split.
split_df.to_csv(OUT_DIR/"video_split_manifest.csv",index=False)
if (SOURCE_SEQ/"group_split_manifest.csv").is_file():
    pd.read_csv(SOURCE_SEQ/"group_split_manifest.csv").to_csv(OUT_DIR/"group_split_manifest.csv",index=False)

class_rows=[]
for cid in sorted(label_map,key=lambda x:int(x)):
    name=label_map[cid]; r={"class_id":int(cid),"label":name}
    for s in ["train","val","test"]:
        yy=np.load(OUT_DIR/f"y_{s}.npy",mmap_mode="r")
        vv=np.load(OUT_DIR/f"videos_{s}.npy",mmap_mode="r")
        mask=(yy==int(cid)); r[f"{s}_windows"]=int(mask.sum()); r[f"{s}_videos"]=int(len(np.unique(vv[mask])))
    class_rows.append(r)
class_support=pd.DataFrame(class_rows)
class_support.to_csv(OUT_DIR/"class_support_by_split.csv",index=False)

metadata={"experiment":"tracked pose sequences using frozen Notebook 1 group split",
    "source_group_split":str(SPLIT_PATH),"tracked_keypoint_dir":str(TRACKED_DIR),"output_dir":str(OUT_DIR),
    "expected_source_videos":len(expected),"tracked_source_videos":len(found),"coverage":len(found)/max(1,len(expected)),
    "is_full_dataset":bool(full_coverage),"complete_source_videos":int(tracked_df["complete_video"].astype(bool).sum()),"partial_source_videos":int((~tracked_df["complete_video"].astype(bool)).sum()),"sequence_length":SEQUENCE_LENGTH,"sequence_stride":SEQUENCE_STRIDE,
    "min_valid_frame_ratio":MIN_VALID_FRAME_RATIO,"min_valid_joints":MIN_VALID_JOINTS,"features_per_frame":51,
    "windows_by_split":counts,"missing_source_videos":len(missing_videos),
    "split_reused_without_changes":True,"note":"A pilot build is not valid for final 101-class training."}
(OUT_DIR/"dataset_metadata.json").write_text(json.dumps(metadata,indent=2),encoding="utf-8")
(NOTEBOOK2_DIR/"data"/"CURRENT_TRACKED_SEQUENCE_DIR.txt").write_text(str(OUT_DIR),encoding="utf-8")
print("DATASET BUILD COMPLETE")
print(json.dumps(metadata,indent=2))
print("Classes with zero windows in any split:")
print(class_support[(class_support.train_windows==0)|(class_support.val_windows==0)|(class_support.test_windows==0)].to_string(index=False))
print("New sequence directory:",OUT_DIR)

## Important

If `is_full_dataset` is false, this is a pilot/partial dataset. Do not claim full UCF101 accuracy or train the full action model from it. The builder writes a new timestamped directory and updates only the Notebook 2 v2 pointer file.